# Component 02 — Retraining on Colab Pro (L4)

**Read `Component_02/COLAB_GUIDE.md` before running this.**

Budget: **~50 minutes, one session, ≤1 GPU-hour.** Run the cells in order and do not skip cell 2 (preflight).

Baseline to beat (audited, reproduced exactly): macro-AUROC **0.9297** · AUPRC **0.7864** · F1 **0.7172**.
Realistic target: **0.935–0.945**. Published PTB-XL work sits at 0.92–0.94.

> This run is an *improvement*, not a dependency. Everything in `RESEARCH_CONTRIBUTION.md` already works on the existing checkpoint.

## 0 · Attach the GPU FIRST

Runtime → Change runtime type → **L4 GPU**, *then* run this. Changing it later restarts the session and you lose the copied data.

In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free',
                      '--format=csv'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > L4 GPU, then re-run.'
print('torch', torch.__version__, '| bf16', torch.cuda.is_bf16_supported())

## 1 · Mount Drive and stage everything on LOCAL disk

`/content/` is a fast local SSD. `/content/drive/` is network storage — training off it would waste GPU time waiting on I/O.

Expects the Drive layout from `COLAB_GUIDE.md`:
```
MyDrive/Component_02/{data,src,train,audit,csv}
```

In [ ]:
import os, shutil, time
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/Component_02'      # <-- your Drive folder
WORK  = '/content/rp'                              # local working copy

missing = [p for p in ['data/train_X.npy','src/models.py','train/train_gpu.py',
                       'csv/train.csv','csv/norm_stats.json']
           if not os.path.exists(f'{DRIVE}/{p}')]
assert not missing, f'Missing in Drive: {missing}\nSee COLAB_GUIDE.md section 2.'
print('Drive layout OK')

In [ ]:
# Build the directory layout the scripts expect, then copy (once).
for d in ['Component_02/src','Component_02/train','Component_02/audit',
          'Component_02/data','Component_02/checkpoints','_archive/data']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)

t0 = time.time()
for sub, dst in [('src','Component_02/src'), ('train','Component_02/train'),
                 ('audit','Component_02/audit'), ('csv','_archive/data')]:
    if os.path.isdir(f'{DRIVE}/{sub}'):
        shutil.copytree(f'{DRIVE}/{sub}', f'{WORK}/{dst}', dirs_exist_ok=True)

# The 2.1 GB of packed arrays — the only slow copy, ~1-2 min.
for f in sorted(os.listdir(f'{DRIVE}/data')):
    src, dst = f'{DRIVE}/data/{f}', f'{WORK}/Component_02/data/{f}'
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f'  {f}  {os.path.getsize(dst)/1e6:.0f} MB')

print(f'\nstaged in {time.time()-t0:.0f}s')
os.chdir(WORK)
!du -sh Component_02/data && df -h /content | tail -1

In [ ]:
!pip -q install wfdb 'scipy>=1.10' 2>/dev/null; echo deps ok

## 2 · PREFLIGHT — do not skip

30 seconds. Verifies the GPU, free VRAM, packed-data integrity, disk space, and runs a **real forward+backward** at your batch size to confirm it fits and to estimate minutes/epoch.

**If it reports FAIL, stop.** Fixing it now costs seconds; finding out at minute 38 costs the session.

In [ ]:
!python Component_02/train/preflight.py --batch 128

## 3 · Train

One run. No sweeps — the hyper-parameters are already chosen.

* `--max-minutes 60` stops cleanly inside budget instead of being killed mid-write.
* CUDA OOM halves the batch and restarts the epoch rather than crashing.
* A checkpoint is written every epoch, atomically.

Watch the `AUROC` column. If it has not passed **0.930 by epoch 20**, stop and send me the log rather than burning the rest.

In [ ]:
!python Component_02/train/train_gpu.py \
    --epochs 40 --batch 128 --lr 3e-3 --seed 0 --workers 2 --max-minutes 60

### 3b · If the session dropped — RESUME, do not restart

Re-run cells 0–1 to remount and re-stage, then this. Optimizer, scheduler, EMA and RNG state are all restored; a disconnect at epoch 30 costs one epoch, not thirty.

In [ ]:
# Copy the checkpoint back from Drive first if you saved it there (cell 6), then:
# !python Component_02/train/train_gpu.py --resume --max-minutes 60

## 4 · Calibration + conformal triage

Fitted on fold 9, verified on fold 10. `--filter` must match training (the packer filters by default, so keep it on).

In [ ]:
!python Component_02/train/fit_calibration.py --model resnet_se --from-logits --preset safety


### 4b · All three operating points (for the trade-off table in your slides)

`--reuse-logits` means each extra point costs seconds, not GPU time.

In [ ]:
import subprocess
for preset in ['balanced', 'throughput', 'safety']:   # 'safety' last = the one that ships
    print('=' * 70); print(preset); print('=' * 70)
    out = subprocess.run(
        ['python', 'Component_02/train/fit_calibration.py', '--model', 'resnet_se',
         '--from-logits', '--preset', preset],
        capture_output=True, text=True).stdout
    keep = False
    for line in out.splitlines():
        if 'DO THE GUARANTEES' in line:
            keep = True
        if keep:
            print(line)
        if 'autonomous subset' in line:
            keep = False


## 5 · Confirm every audit fix still holds (26 checks)

In [ ]:
# This suite replays RAW-millivolt adversarial inputs through the quality gate,
# so it needs _archive/data/signals_cache, which is NOT uploaded to Colab.
# The packed arrays are already filtered and normalised and cannot substitute.
#
# Run it on your LAPTOP after downloading the checkpoint. In PowerShell:
#
#   $env:ECG_CKPT='Component_02/checkpoints/best_model.pt'
#   $env:ECG_MODEL='resnet_se'
#   $env:ECG_FILTER='1'
#   python -X utf8 Component_02/audit/08_verify_fixes.py

print('Skip on Colab. Run 08_verify_fixes.py on your laptop -- see comment above.')


## 6 · Save results back to Drive, then download

Three small files are all you need on your laptop. **Run this cell even if training is still going** — it also backs up `last.pt` so a disconnect is recoverable.

In [ ]:
import glob
out = f'{DRIVE}/checkpoints'
os.makedirs(out, exist_ok=True)
for f in glob.glob('Component_02/checkpoints/*'):
    if os.path.isfile(f):
        shutil.copy(f, f'{out}/{os.path.basename(f)}')
        print(f'-> Drive: {os.path.basename(f)}  {os.path.getsize(f)/1e6:.1f} MB')

from google.colab import files
for f in ['best_model.pt', 'calibrator.json', 'conformal_triage.json']:
    p = f'Component_02/checkpoints/{f}'
    if os.path.exists(p):
        files.download(p)

## 7 · Back on your laptop

Put the three downloaded files in `Component_02/checkpoints/`, then:

```bash
ECG_MODEL=resnet_se ECG_FILTER=1 \
ECG_CKPT=Component_02/checkpoints/best_model.pt \
python -X utf8 Component_02/app/app.py
```

---

### Compute-unit discipline

| Do | Don't |
|---|---|
| Run preflight every session | Start training without it |
| One run, 40 epochs | Hyper-parameter sweeps |
| `--resume` after a disconnect | Restart from epoch 1 |
| Train from `/content/` | Train from `/content/drive/` |
| Extra seeds only if units remain | Three seeds before you have one working run |

**Honest note:** if you land at 0.931 rather than 0.940, report it. The conformal contribution does not depend on beating the baseline, and an honest null result is better science than tuning on the test set until the number moves.